In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Annotated
import operator
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
# =========================================================
# 1. TASK / WORKER MODEL
# =========================================================
class Worker(BaseModel):
    name: str
    dependencies: list[str] = Field(
        default_factory=list
    )

    status: str = "PENDING"

    # =====================================================
    # MISTAKE FIX #1
    # =====================================================
    # Tumhare old code mein retry_count nahi tha.
    #
    # Retry count function ke andar nahi hona chahiye,
    # because function har invocation par variable ko reset
    # kar deta hai.
    #
    # Isliye retry information task ke andar store kar rahe hain.
    # =====================================================

    retry_count: int = 0

    max_retries: int = 2


# =========================================================
# 2. GLOBAL STATE
# =========================================================

class NodeState(BaseModel):

    workers: list[Worker] = Field(
        default_factory=list
    )

    completed_tasks: Annotated[
        list[str],
        operator.add
    ] = Field(
        default_factory=list
    )

    failed_tasks: Annotated[
        list[str],
        operator.add
    ] = Field(
        default_factory=list
    )


# =========================================================
# 3. CREATE TASKS
# =========================================================

def create_tasks(state: NodeState):

    workers = [

        # ---------------------------------------------
        # Research has no dependency
        # ---------------------------------------------

        Worker(
            name="Research",
            dependencies=[]
        ),

        # ---------------------------------------------
        # FinanceReview has no dependency
        # ---------------------------------------------
        #
        # We will intentionally make it fail in Runtime
        # so that we can test the Day 44 retry mechanism.
        #
        # IMPORTANT:
        # We are NOT setting status="FAILED" here.
        #
        # Runtime will simulate the failure.
        # ---------------------------------------------

        Worker(
            name="FinanceReview",
            dependencies=[]
        ),

        # ---------------------------------------------
        # FinalReview depends on BOTH tasks
        # ---------------------------------------------

        Worker(
            name="FinalReview",
            dependencies=[
                "Research",
                "FinanceReview"
            ]
        ),

        # ---------------------------------------------
        # Writer depends on FinalReview
        # ---------------------------------------------

        Worker(
            name="Writer",
            dependencies=[
                "FinalReview"
            ]
        )
    ]

    print("\n========== TASKS CREATED ==========\n")

    for task in workers:

        print(
            f"{task.name:15} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status} | "
            f"Retries: {task.retry_count}/{task.max_retries}"
        )

    return {
        "workers": workers
    }


# =========================================================
# 4. SCHEDULER
# =========================================================

def scheduler(state: NodeState):

    print("\n========== SCHEDULER ==========\n")

    ready_tasks = []

    for task in state.workers:

        # =================================================
        # ALREADY COMPLETED
        # =================================================
        #
        # Completed task ko dobara execute nahi karna.
        # =================================================

        if task.name in state.completed_tasks:
            continue

        # =================================================
        # FAILED TASK
        # =================================================
        #
        # Scheduler failed task ko directly execute nahi
        # karega.
        #
        # Pehle Retry Router decide karega ke retry karna hai
        # ya HITL par jana hai.
        # =================================================

        if task.status == "FAILED":
            continue

        # =================================================
        # DEPENDENCY CHECK
        # =================================================

        dependencies_satisfied = all(
            dependency in state.completed_tasks
            for dependency in task.dependencies
        )

        # =================================================
        # READY
        # =================================================

        if dependencies_satisfied:

            task.status = "READY"

            ready_tasks.append(task)

        # =================================================
        # BLOCKED
        # =================================================

        else:

            task.status = "BLOCKED"

    # =====================================================
    # CONCURRENCY LIMIT
    # =====================================================

    concurrency_limit = 2

    running_tasks = ready_tasks[:concurrency_limit]

    waiting_tasks = ready_tasks[concurrency_limit:]

    # =====================================================
    # RUNNING
    # =====================================================

    for task in running_tasks:

        task.status = "RUNNING"

    # =====================================================
    # WAITING
    # =====================================================

    for task in waiting_tasks:

        task.status = "WAITING"

    # =====================================================
    # PRINT CURRENT SCHEDULER STATE
    # =====================================================

    for task in state.workers:

        print(
            f"{task.name:15} | "
            f"Dependencies: {task.dependencies} | "
            f"Status: {task.status} | "
            f"Retries: {task.retry_count}/{task.max_retries}"
        )

    return {
        "workers": state.workers
    }


# =========================================================
# 5. RUNTIME
# =========================================================

def runtime(state: NodeState):

    print("\n========== RUNTIME ==========\n")

    completed_now = []

    failed_now = []

    for task in state.workers:

        # =================================================
        # MISTAKE FIX #2
        # =================================================
        #
        # Tumhare old code mein:
        #
        # task.status = "SUCCESS"
        #
        # aur uske BAAD:
        #
        # if task.status == "FAILED":
        #
        # tha.
        #
        # Ye logically impossible tha because status already
        # SUCCESS ban chuka tha.
        #
        # Ab pehle decide karenge task SUCCESS hai ya FAILED.
        # =================================================

        if task.status != "RUNNING":
            continue
        print(
            f"Executing: {task.name}"
        )
        # =================================================
        # SIMULATED FAILURE
        # =================================================
        #
        # Day 44 testing ke liye FinanceReview intentionally
        # fail hoga.
        #
        # Real project mein yahan actual worker/API/LLM
        # execution hoga.
        # =================================================

        if task.name == "FinanceReview":

            task.status = "FAILED"

            failed_now.append(
                task.name
            )

            print(
                f"{task.name} -> FAILED"
            )
        else:
            task.status = "SUCCESS"
            completed_now.append(
                task.name
            )
            print(
                f"{task.name} -> SUCCESS"
            )
    return {
        "workers": state.workers,

        "completed_tasks": completed_now,

        "failed_tasks": failed_now
    }
# =========================================================
# 6. ROUTER AFTER RUNTIME
# =========================================================
def route_after_runtime(state: NodeState):
    # =====================================================
    # MISTAKE FIX #3
    # =====================================================
    #
    # Tumhare old code mein Runtime se multiple independent
    # conditional edges banayi gayi thi:
    #
    # Runtime -> check_remaining_tasks
    # Runtime -> retry
    # Runtime -> hitl
    #
    # Better architecture:
    #
    # Runtime
    #    ↓
    # One Router
    #    ├── RETRY
    #    ├── HITL
    #    └── SCHEDULER
    #
    # =====================================================
    for task in state.workers:
        if task.status == "FAILED":
            # =============================================
            # RETRY AVAILABLE
            # =============================================
            if task.retry_count < task.max_retries:
                return "RETRY"
            # =============================================
            # RETRIES EXHAUSTED
            # =============================================
            return "HITL"

    # =====================================================
    # NO FAILURE
    # =====================================================

    for task in state.workers:

        if task.name not in state.completed_tasks:

            return "SCHEDULER"

    return "END"


# =========================================================
# 7. RETRY HANDLER
# ========================================================
def retry_handler(state: NodeState):

    print("\n========== RETRY HANDLER ==========\n")

    for task in state.workers:

        if task.status != "FAILED":
            continue
        if task.retry_count < task.max_retries:
                task.retry_count += 1
                task.status = "READY"
        else:
         task.status = "HITL_READY"
        print(
                f"{task.name} -> RETRY "
                f"{task.retry_count}/{task.max_retries}"
            )

    return {
        "workers": state.workers
    }


# =========================================================
# 8. HITL
# =========================================================

def hitl(state: NodeState):

    print("\n========== HITL ==========\n")

    for task in state.workers:
        if task.status != "HITL_READY":
            continue
   
        # -------------------------------------------------
        # Day 44 ke liye simple simulation
        # -------------------------------------------------
        #
        # Real LangGraph HITL mein yahan interrupt()
        # use kar sakte ho.
        #
        # Abhi hum learning ke liye direct decision simulate
        # kar rahe hain.
        # -------------------------------------------------
        human_decision = interrupt(
    "Choose RETRY, APPROVE or REJECT"
)
        if human_decision == "RETRY":

            task.status = "READY"
            return "Scheduler"

            print(
                f"{task.name} -> HUMAN REQUESTED RETRY"
            )

        elif human_decision == "APPROVE":

            task.status = "SUCCESS"
            state.completed_tasks.append(task.name)

            print(
                f"{task.name} -> HUMAN APPROVED"
            )

        elif human_decision == "REJECT":

            task.status = "REJECTED"

            print(
                f"{task.name} -> HUMAN REJECTED"
            )

    return {
        "workers": state.workers
    }


# =========================================================
# 9. CHECK REMAINING TASKS
# =========================================================

def check_remaining_tasks(state: NodeState):

    # =====================================================
    # Rejected task ko workflow ko endlessly loop karne se
    # rokna zaroori hai.
    # =====================================================

    for task in state.workers:

        if task.status in [
            "REJECTED"
        ]:

            return "END"

    # =====================================================
    # Agar koi task abhi complete nahi hua
    # =====================================================

    for task in state.workers:

        if task.name not in state.completed_tasks:

            return "SCHEDULER"

    return "END"


# =========================================================
# 10. BUILD GRAPH
# =========================================================

graph = StateGraph(NodeState)


graph.add_node(
    "CreateTasks",
    create_tasks
)

graph.add_node(
    "Scheduler",
    scheduler
)

graph.add_node(
    "Runtime",
    runtime
)

graph.add_node(
    "Retry",
    retry_handler
)

graph.add_node(
    "HITL",
    hitl
)


# =========================================================
# 11. GRAPH FLOW
# =========================================================

graph.add_edge(
    START,
    "CreateTasks"
)

graph.add_edge(
    "CreateTasks",
    "Scheduler"
)

graph.add_edge(
    "Scheduler",
    "Runtime"
)


# =========================================================
# 12. RUNTIME ROUTER
# =========================================================

graph.add_conditional_edges(

    "Runtime",

    route_after_runtime,

    {
        "RETRY": "Retry",

        "HITL": "HITL",

        "SCHEDULER": "Scheduler",

        "END": END
    }
)


# =========================================================
# 13. RETRY → SCHEDULER
# =========================================================

graph.add_edge(
    "Retry",
    "Scheduler"
)


# =========================================================
# 14. HITL ROUTING
# =========================================================

graph.add_conditional_edges(

    "HITL",

    check_remaining_tasks,

    {
        "SCHEDULER": "Scheduler",

        "END": END
    }
)


# =========================================================
# 15. COMPILE
# =========================================================

app = graph.compile(checkpointer=checkpointer)


# =========================================================
# 16. EXECUTE
# =========================================================
config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}
result = app.invoke(
    {},
    config=config
)
result = app.invoke(
    Command(resume="RETRY"),
    config=config
)


# =========================================================
# 17. FINAL STATE
# =========================================================

print("\n========== FINAL STATE ==========\n")

print(
    "Completed Tasks:",
    result["completed_tasks"]
)

print(
    "Failed Tasks:",
    result.get("failed_tasks", [])
)
print()
for task in result["workers"]:

    print(
        f"Task: {task.name}"
    )

    print(
        f"Dependencies: {task.dependencies}"
    )

    print(
        f"Status: {task.status}"
    )

    print(
        f"Retry Count: "
        f"{task.retry_count}/{task.max_retries}"
    )

    print(
        "-----------------------------------"
    )